# Kestrel-Nano on Colab — 4k context → SFT → Roost

All heavy compute for Kestrel runs here, not on the home PC.

**Before running:** `Runtime → Change runtime type → A100 GPU`, signed in as **malxshrouds@gmail.com** (the account that owns the Kestrel Drive folder and the Colab Pro units).

Then `Runtime → Run all`. Every stage is **resumable**: if the session disconnects, reconnect and `Run all` again — finished stages are skipped and training resumes from its last checkpoint on Drive.

| stage | what | est. A100 time |
|---|---|---|
| A | context eval of v0.2 (baseline) | ~10 min |
| B | **context extension 1024 → 4096** (1,000 steps, 0.2B tokens) | ~4–7 h |
| C | evals after extension (4k positions + 1k regression check) | ~20 min |
| D | **SFT** (23,779 examples, 2 epochs, seq 4096) | ~1 h |
| E | chat samples | ~2 min |
| F | **Roost teach-me-today** (50 novel facts) | ~15 min |

Times are estimates — the A100 rate for Kestrel has never been measured. Stage B prints real tok/s within a minute.

Outputs all land in Drive under `Kestrel/runs`, `Kestrel/evals` and `Kestrel/roost`. Training writes `status.json` there, so progress is readable from the home PC through Google Drive for desktop, and the KestrelMonitor app can be pointed at it.

In [1]:
# 0. GPU check — refuse to burn units on a card that is no faster than the home RTX 3060
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'torch {torch.__version__} | {name} | {mem:.0f} GB')
if 'A100' not in name and 'H100' not in name:
    raise SystemExit(f'Got {name}, not an A100. L4/T4 are no faster than the home 3060 for Kestrel '
                     'and would waste compute units. Runtime -> Change runtime type -> A100, then rerun.')

torch 2.11.0+cu130 | NVIDIA A100-SXM4-80GB | 85 GB


In [2]:
# 1. Drive (must be malxshrouds@gmail.com — that is where Kestrel/ lives)
import os
from google.colab import drive
drive.mount('/content/drive')
KD = '/content/drive/MyDrive/Kestrel'
assert os.path.isdir(KD), f'{KD} not found - mounted the wrong Google account?'
for d in ('runs', 'evals', 'roost'):
    os.makedirs(f'{KD}/{d}', exist_ok=True)
print('Drive OK:', sorted(os.listdir(KD)))

ValueError: mount failed

In [ ]:
# 2. Code. The repo is public, so no credentials. Colab's preinstalled torch is used
#    as-is: requirements.txt pins torch<2.8 for the old GTX 1080 and must NOT be installed here.
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers
import tokenizers; print('tokenizers', tokenizers.__version__)

In [ ]:
# 3. Data -> local disk. Training reads random windows constantly; doing that over the
#    Drive mount would be far too slow, so copy once per session (~minutes).
#
#    Completeness check: the home PC uploads FILES.json LAST, after every shard. If it is
#    missing, the upload is still in progress - and running anyway would make the
#    curriculum loader silently train on whatever subset of domains had arrived.
import shutil, glob, json
CORPUS = f'{KD}/datasets/corpus-5b'
if not os.path.exists(f'{CORPUS}/FILES.json'):
    raise SystemExit('corpus upload not finished yet (FILES.json is written last). '
                     'Wait for the home PC upload to complete, then rerun.')
def sync(src, dst):
    os.makedirs(dst, exist_ok=True)
    n = 0
    for f in sorted(glob.glob(f'{src}/*')):
        out = f'{dst}/{os.path.basename(f)}'
        if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
            shutil.copy(f, out); n += 1
    gb = sum(os.path.getsize(p) for p in glob.glob(f'{dst}/*')) / 1e9
    print(f'{src} -> {dst}: {n} copied, {gb:.2f} GB present')
sync(CORPUS, '/content/data_5b')
sync(f'{KD}/datasets/sft', '/content/data_sft')
# utf-8-sig: FILES.json is written by Windows PowerShell 5.1, which adds a byte-order mark
expected = json.load(open(f'{CORPUS}/FILES.json', encoding='utf-8-sig'))
bad = [e['name'] for e in expected
       if not os.path.exists(f"/content/data_5b/{e['name']}")
       or os.path.getsize(f"/content/data_5b/{e['name']}") != e['bytes']]
assert not bad, f'corpus incomplete or truncated: {bad}'
print(f'corpus verified: {len(expected)} files match FILES.json')
assert os.path.exists('/content/data_sft/train_tokens.bin'), 'SFT set missing from Drive Kestrel/datasets/sft'
V02 = f'{KD}/checkpoints/v0.2-final/ckpt.pt'
assert os.path.exists(V02), 'v0.2 checkpoint missing from Drive'

In [ ]:
# helpers: stage markers so `Run all` skips finished work after a disconnect
import json, time
def done(name):  return os.path.exists(f'{KD}/runs/{name}.DONE')
def mark(name):  open(f'{KD}/runs/{name}.DONE', 'w').write(time.strftime('%Y-%m-%d %H:%M:%S'))
def finished(run_dir):
    p = f'{run_dir}/status.json'
    return os.path.exists(p) and json.load(open(p)).get('state') == 'done'
CTX = f'{KD}/evals/context_eval.json'
GEN = f'{KD}/evals/generalization.json'

In [ ]:
# A. Baseline: what v0.2 does past position 1024 today, with and without zero-shot interpolation
if not done('A_baseline'):
    !python -m scripts.eval_context --ckpt {V02} --data-dir /content/data_5b --label 'v0.2 @4096, no extension' --out {CTX}
    !python -m scripts.eval_context --ckpt {V02} --data-dir /content/data_5b --rope-scale 4 --label 'v0.2 @4096, PI x4 zero-shot' --out {CTX}
    mark('A_baseline')
else:
    print('A done - skipping')

In [ ]:
# B. Context extension 1024 -> 4096 by position interpolation (rope_scale=4) + 1,000 steps
#    at seq 4096. Resumable: on reconnect it continues from the last checkpoint on Drive.
#    Same 196,608 tokens/step as Phase D. LR is a quarter of pretraining: this adapts
#    positions, it should not move the weights far. Curriculum held on the anneal mix
#    (curriculum-start 0.85) so the data distribution matches what v0.2 finished on.
#    Pause/stop from the home PC by editing Kestrel/runs/ctx4k/control.json on Drive.
RUN_CTX = f'{KD}/runs/ctx4k'
os.makedirs(RUN_CTX, exist_ok=True)
if not os.path.exists(f'{RUN_CTX}/ckpt.pt'):
    print('seeding from v0.2-final ...'); shutil.copy(V02, f'{RUN_CTX}/ckpt.pt')
if not finished(RUN_CTX):
    !python -u -m kestrel.train --preset nano --data-dir /content/data_5b --curriculum --curriculum-start 0.85 \
        --bf16 --grad-checkpoint --fused-adam --prefetch 3 --seq 4096 --batch 8 --accum 6 \
        --steps 19590 --warmup 50 --decay-frac 0.3 --muon-lr 0.005 --adamw-lr 0.00075 \
        --rope-scale 4 --out {RUN_CTX} --log-every 10 --eval-every 100 --eval-iters 6 \
        --ckpt-every 200 --ckpt-secs 1800
print('ctx4k finished:', finished(RUN_CTX))

In [ ]:
# C. Did it work? (1) loss by position out to 4096, (2) regression check at 1024 against
#    v0.2's 1.770 val_mean - the extension must not cost short-context quality.
assert finished(RUN_CTX), 'stage B has not finished - rerun it first'
if not done('C_ctx_eval'):
    !python -m scripts.eval_context --ckpt {RUN_CTX}/ckpt.pt --data-dir /content/data_5b --label 'ctx4k (after extension)' --out {CTX}
    !python -m scripts.eval_generalization --ckpt {RUN_CTX}/ckpt.pt --data-dir /content/data_5b --seq 1024 --label 'ctx4k @seq1024' --out {GEN}
    !python -m scripts.eval_generalization --ckpt {RUN_CTX}/ckpt.pt --data-dir /content/data_5b --seq 4096 --label 'ctx4k @seq4096' --out {GEN}
    mark('C_ctx_eval')
else:
    print('C done - skipping')

In [ ]:
# D. SFT on the 4k model. train.py reads the RoPE scale from the checkpoint, but it is
#    ALSO passed explicitly here: if it were silently lost, SFT would train a 4k model
#    at 1k positions and nothing would error. The data is built for seq 4096.
#    23,779 examples / 16.2M tokens; 2 epochs ~= 165 steps at 196,608 tokens/step.
#    Loss is scored on Assistant tokens only (the mask in data_sft).
RUN_SFT = f'{KD}/runs/sft'
os.makedirs(RUN_SFT, exist_ok=True)
if not os.path.exists(f'{RUN_SFT}/ckpt.pt'):
    print('seeding from ctx4k ...'); shutil.copy(f'{RUN_CTX}/ckpt.pt', f'{RUN_SFT}/ckpt.pt')
if not finished(RUN_SFT):
    !python -u -m kestrel.train --preset nano --sft-dir /content/data_sft \
        --bf16 --grad-checkpoint --fused-adam --prefetch 3 --seq 4096 --batch 8 --accum 6 \
        --steps 19755 --warmup 15 --decay-frac 0.5 --muon-lr 0.004 --adamw-lr 0.0006 \
        --rope-scale 4 --out {RUN_SFT} --log-every 5 --eval-every 25 --eval-iters 10 \
        --ckpt-every 50 --ckpt-secs 1800
print('sft finished:', finished(RUN_SFT))

In [ ]:
# E. Talk to it. Same template it was trained on. Saved to Drive so it can be read from home.
import torch, sys
sys.path.insert(0, REPO)
from kestrel.generate import load, generate
model, tok, cfg, seen = load(f'{RUN_SFT}/ckpt.pt', 'tokenizer/kestrel-bpe.json', torch.device('cuda'))
print(f'rope_scale={cfg.rope_scale}  tokens_seen={seen/1e9:.3f}B')
assert cfg.rope_scale == 4, 'SFT checkpoint lost its 4k position setting'
prompts = ['What AI model are you?', 'Who made you?', 'Write a Python function that reverses a string.',
           'List all .log files modified in the last 7 days.', 'What is the capital of Vantersk?',
           'Explain what a git rebase does in two sentences.']
out = []
for p in prompts:
    text = generate(model, tok, f'User: {p}\n\nAssistant:', max_new=160, temperature=0.7,
                    ctx=4096, device='cuda')
    ans = text.split('Assistant:', 1)[-1].strip()
    out.append({'prompt': p, 'answer': ans})
    print(f'\n>>> {p}\n{ans}')
json.dump(out, open(f'{KD}/evals/chat_samples.json', 'w'), indent=1)
del model; torch.cuda.empty_cache()

In [ ]:
# F. Roost teach-me-today (S3 criterion): teach 50 novel facts about nonsense entities,
#    run the eval-gated consolidation, report recall and regression. Accepted only if
#    recall >= 60% AND old-capability regression <= 2%; otherwise it rolls back.
#    Done = consolidation.jsonl exists (a shell `!` failure does not raise, so a DONE
#    marker alone would hide a crash). A crashed attempt leaves the 50 facts in the
#    store; clear it so a rerun teaches 50, not 100.
ROOST_LOG = f'{KD}/roost/consolidation.jsonl'
ROOST_STORE = f'{KD}/roost/episodic.jsonl'
if not os.path.exists(ROOST_LOG):
    if os.path.exists(ROOST_STORE):
        os.remove(ROOST_STORE)
    !python -m scripts.roost teach-me-today --ckpt {RUN_SFT}/ckpt.pt --replay-dir /content/data_5b --store {ROOST_STORE} --n 50 --out {KD}/roost/consolidated_ckpt.pt
else:
    print('F done - skipping')
!cat {ROOST_LOG}
